# Experiment: teaching a 0.6B model the answer contract with LoRA

**This is an experiment, not part of the shipped pipeline.** The pipeline answers with SmolLM3 3B on llama.cpp; nothing here changes it.

**Question.** The pipeline's prompt asks for an answer contract: cite each source as `[n]`, and when the sources don't hold the answer, reply with exactly *I cannot answer this from the provided documents.* Can a LoRA adapter, trained on a few hundred examples generated automatically from the corpus, teach that contract to a model too small to follow it from the prompt alone? And does the gain carry over to the eval questions, none of which it trained on?

**Held fixed:** retrieval (hybrid search, top 5), the prompt (`grounded-v2`), the graders and the three eval sets. Only the model changes: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) (Apache 2.0) without and with the adapter, greedy decoding, reasoning mode off.

**How the training data avoids the eval** (details in `bankrag/finetune_data.py`, built by `python -m bankrag finetune-data`):
1. Only chunks from articles that no eval question is about.
2. SmolLM3 writes one question, answer and supporting quote per chunk; code drops it unless the quote is in the chunk word for word and the question stands on its own.
3. Each question becomes an *answer* example (its chunk is among the five retrieved sources; target = answer + `[n]`) and, for 60% of questions, a *refuse* example (every chunk of that article removed from the sources; target = the refusal sentence). Same question, with and without the evidence, so the model has to look at the sources.

In [1]:
import json
import math
import random
import sys
import time
from collections import Counter
from pathlib import Path

if Path.cwd().name == "notebooks":
    sys.path.insert(0, str(Path.cwd().parent))

import peft
import torch
import transformers
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

from bankrag import finetune_data as fd
from bankrag import hf_chat
from bankrag.config import LORA_ADAPTER_DIR
from bankrag.evaluate import run_eval
from bankrag.grading import load_questions, normalise
from bankrag.index import build_index
from bankrag.report import latest_runs

MODEL_ID = "Qwen/Qwen3-0.6B"
PROMPT = "grounded-v2"
SETS = ["main", "holdout", "plain"]
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}), transformers {transformers.__version__}, peft {peft.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

D:\Projects\BankingRAGEval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1111 17:03:38.728000 37900 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


torch 2.11.0+cu128 (CUDA 12.8), transformers 5.22.2, peft 0.21.2
GPU: NVIDIA GeForce RTX 3060 Ti, 8.0 GiB


## 1. Training data

In [2]:
generated = fd.load_jsonl(fd.GENERATED_FILE)
examples = fd.load_jsonl(fd.EXAMPLES_FILE)
print(f"{len(generated)} chunks shown to the generator, from {len({g['article_id'] for g in generated})} articles")
for reason, n in Counter(g["rejected"] or "kept" for g in generated).most_common():
    print(f"  {n:4}  {reason}")
print()
for (split, kind), n in sorted(Counter((e["split"], e["kind"]) for e in examples).items()):
    print(f"  {split:5} {kind:6} {n}")

# Leak checks: no training example comes from an eval question's gold article, and no
# training question repeats an eval question.
gold = fd.eval_gold_articles()
sources = {e["id"].rsplit("-", 2)[0] for e in examples}
assert not sources & gold, sources & gold
eval_questions = {normalise(q.question) for s in SETS for q in load_questions(s)}
assert not eval_questions & {normalise(e["question"]) for e in examples}
print(f"\nleak checks passed: {len(sources)} source articles, none of the {len(gold)} eval gold articles")

317 chunks shown to the generator, from 164 articles
   224  kept
    52  quote not in the passage
    20  answer length
    18  question refers to the passage or an article number
     3  question shape

  train answer 188
  train refuse 114
  val   answer 26
  val   refuse 14

leak checks passed: 137 source articles, none of the 34 eval gold articles


In [3]:
def words(text):
    return set(normalise(text).replace("?", "").split())

# The training questions most similar to an eval question, by word overlap (Jaccard).
pairs = []
for q in (q for s in SETS for q in load_questions(s)):
    best = max(examples, key=lambda e: len(words(e["question"]) & words(q.question)) / len(words(e["question"]) | words(q.question)))
    sim = len(words(best["question"]) & words(q.question)) / len(words(best["question"]) | words(q.question))
    pairs.append((sim, q.question, best["question"]))
for sim, q, t in sorted(pairs, reverse=True)[:3]:
    print(f"{sim:.2f}  eval:  {q}\n      train: {t}\n")

0.48  eval:  In what format does a data subject have the right to receive their data under the right to data portability?
      train: Under the GDPR, what right does a data subject have to mandate a not-for-profit body to lodge a complaint?

0.45  eval:  What is the longest notice period a payment service user can be required to give to terminate a framework contract?
      train: Under PSD2, what is the right of the payment service user to terminate the framework contract?

0.43  eval:  What is the deadline for a payment service provider to reply to a customer's complaint?
      train: Under PSD2, what is the condition for a payer to have the right to use a payment initiation service provider?



In [4]:
chat = fd.to_chat(examples)
for kind in ("answer", "refuse"):
    ex = next(c for c in chat if c.kind == kind)
    print(f"--- {kind} example: {ex.id}")
    print(ex.user[-700:])
    print(f"TARGET: {ex.target}\n")

--- answer example: PSD2-81-1-answer
quest;
(b) the payer has given explicit consent to the account servicing payment service provider to respond to requests from a specific payment service provider to confirm that the amount corresponding to a certain card-based payment transaction is available on the payer’s payment account;
(c) the consent referred to in point (b) has been given before the first request for confirmation is made.
2. The payment service provider may request the confirmation referred to in paragraph 1 where all of the following conditions are met:

Question: Under PSD2, what must a payment service provider ensure when charges other than those referred to in paragraph 2 are deducted from the amount transferred?

TARGET: The payment service provider of the payer shall ensure that the payee receives the full amount of the payment transaction initiated by the payer. [1]

--- refuse example: PSD2-68-2-refuse
ficient channels;
(c) ensure that any other information about the 

## 2. The base model on the eval sets

In [5]:
index = build_index()
base = hf_chat.load(MODEL_ID)
runs = {}
for s in SETS:
    runs[("base", s)] = run_eval("qwen3-0.6b", PROMPT, index=index, question_set=s, llm=base, progress=False)
    print(f"base {s:8} answer accuracy {runs[('base', s)].summary['answer_accuracy']:.3f}")
del base
torch.cuda.empty_cache()

D:\Projects\BankingRAGEval\.venv\Lib\site-packages\huggingface_hub\utils\tqdm.py:176: UserWarning: Cannot enable progress bars: environment variable `HF_HUB_DISABLE_PROGRESS_BARS=1` is set and has priority.
  warnings.warn(


base main     answer accuracy 0.758


base holdout  answer accuracy 0.636


base plain    answer accuracy 0.375


## 3. LoRA training

Rank 16 on every attention and MLP projection, 2 epochs, learning rate 2e-4 with cosine decay, an effective batch of 8 examples. The loss counts only the answer tokens, not the prompt. The prompt tokens are made by the same function the evaluation uses, so the adapter is trained on exactly the prompt it will see.

The model scores its whole 152,000-token vocabulary at every position, and a prompt here is about 1,100 tokens long, so computing those scores for every position takes several GiB. Only the answer positions are needed for the loss, so `logits_to_keep` asks for just those. That gives the same loss (checked on one example: 0.7906 both ways) and fits the 8 GB card.

In [6]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
END = tok("<|im_end|>\n", add_special_tokens=False)["input_ids"]  # Qwen's end of turn


def encode(ex):
    prompt = hf_chat.chat_prompt_ids(tok, ex.system, ex.user)
    target = tok(ex.target, add_special_tokens=False)["input_ids"] + END
    return {"input_ids": prompt + target, "prompt_len": len(prompt), "id": ex.id}


train = [encode(c) for c in chat if c.split == "train"]
val = [encode(c) for c in chat if c.split == "val"]
lengths = sorted(len(e["input_ids"]) for e in train)
print(f"{len(train)} training and {len(val)} validation examples; "
      f"length median {lengths[len(lengths) // 2]}, max {lengths[-1]} tokens")

302 training and 40 validation examples; length median 1085, max 1387 tokens


In [7]:
random.seed(0)
torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


In [8]:
EPOCHS, LR, ACCUM = 2, 2e-4, 8


def answer_loss(ex):
    # Mean cross-entropy over the answer tokens, with scores computed only where they are needed.
    ids = torch.tensor([ex["input_ids"]], device="cuda")
    p = ex["prompt_len"]
    logits = model(input_ids=ids, logits_to_keep=ids.shape[1] - p + 1).logits[0, :-1].float()
    return torch.nn.functional.cross_entropy(logits, ids[0, p:])


def mean_loss(data):
    model.eval()
    with torch.no_grad():
        total = sum(answer_loss(ex).item() for ex in data)
    model.train()
    return total / len(data)


params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
steps = math.ceil(len(train) / ACCUM) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, steps // 20), steps)
torch.cuda.reset_peak_memory_stats()
history = [{"epoch": 0, "step": 0, "train_loss": None, "val_loss": mean_loss(val)}]
print(f"before training: validation loss {history[0]['val_loss']:.3f}")

t0, step = time.perf_counter(), 0
model.train()
for epoch in range(1, EPOCHS + 1):
    order = list(range(len(train)))
    random.Random(epoch).shuffle(order)
    running, seen = 0.0, 0
    for i, j in enumerate(order, 1):
        loss = answer_loss(train[j])
        (loss / ACCUM).backward()
        running, seen = running + loss.item(), seen + 1
        if i % ACCUM == 0 or i == len(order):
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step += 1
    history.append({"epoch": epoch, "step": step, "train_loss": running / seen, "val_loss": mean_loss(val)})
    print(f"epoch {epoch}: train loss {running / seen:.3f}, validation loss {history[-1]['val_loss']:.3f}, "
          f"{time.perf_counter() - t0:.0f} s")

print(f"\n{step} optimizer steps in {time.perf_counter() - t0:.0f} s, "
      f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
model.save_pretrained(LORA_ADAPTER_DIR)
RESULTS = Path(fd.__file__).resolve().parent.parent / "results" / "lora"
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "training.json").write_text(json.dumps({
    "model": MODEL_ID, "lora": {"r": lora.r, "alpha": lora.lora_alpha, "dropout": lora.lora_dropout,
                                "target_modules": sorted(lora.target_modules)},
    "epochs": EPOCHS, "learning_rate": LR, "grad_accum": ACCUM,
    "train_examples": len(train), "val_examples": len(val), "optimizer_steps": step,
    "seconds": round(time.perf_counter() - t0, 1),
    "peak_gpu_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2), "history": history,
}, indent=2), encoding="utf-8")
print(f"adapter saved to {LORA_ADAPTER_DIR.relative_to(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())}")

before training: validation loss 0.858


epoch 1: train loss 0.250, validation loss 0.212, 217 s


epoch 2: train loss 0.090, validation loss 0.209, 427 s

76 optimizer steps in 427 s, peak GPU memory 1.9 GiB


adapter saved to .cache\lora\qwen3-0.6b-contract


## 4. The model with the adapter on the eval sets

In [9]:
# Fold the adapter into the weights, so answering is as fast as with the base model.
merged = model.merge_and_unload()
merged.gradient_checkpointing_disable()
merged.config.use_cache = True
merged.eval()
tuned = hf_chat.HFChatModel(model=merged, tokenizer=tok)
for s in SETS:
    runs[("lora", s)] = run_eval("qwen3-0.6b-lora", PROMPT, index=index, question_set=s, llm=tuned, progress=False)
    print(f"lora {s:8} answer accuracy {runs[('lora', s)].summary['answer_accuracy']:.3f}")

lora main     answer accuracy 0.576


lora holdout  answer accuracy 0.455


lora plain    answer accuracy 0.125


In [10]:
def cited_share(run):
    """Of the answers that aren't refusals, the share that cite at least one valid source."""
    answered = [r for r in run.rows if not r["refused"]]
    return sum(r["citation_valid"] for r in answered) / len(answered) if answered else float("nan")


smol = latest_runs()
rows = []
for s in SETS:
    for name, summary, cited in (
        ("Qwen3 0.6B", runs[("base", s)].summary, cited_share(runs[("base", s)])),
        ("Qwen3 0.6B + LoRA", runs[("lora", s)].summary, cited_share(runs[("lora", s)])),
        ("SmolLM3 3B (reference)", smol[("local", PROMPT, s)]["summary"], None),
    ):
        refusal = f"{summary['refusal_rate']:.0%}" if summary["n_unanswerable"] else "-"
        rows.append(
            f"| {s} | {name} | {summary['answer_accuracy']:.0%} | {summary['citation_accuracy']:.0%} | {refusal} "
            f"| {summary['false_refusal_rate']:.0%} | {'-' if cited is None else f'{cited:.0%}'} "
            f"| {summary['tokens_out_mean']:.0f} | {summary['latency_ms_p50'] / 1000:.2f} s |"
        )
print("| Set | Model | Answer accuracy | Cites the right article | Refuses when it should | Wrongly refuses "
      "| Answers with a valid citation | Tokens out | Median latency |")
print("|---|---|---|---|---|---|---|---|---|")
print("\n".join(rows))

| Set | Model | Answer accuracy | Cites the right article | Refuses when it should | Wrongly refuses | Answers with a valid citation | Tokens out | Median latency |
|---|---|---|---|---|---|---|---|---|
| main | Qwen3 0.6B | 76% | 94% | 50% | 15% | 91% | 83 | 4.63 s |
| main | Qwen3 0.6B + LoRA | 58% | 64% | 100% | 36% | 100% | 23 | 0.70 s |
| main | SmolLM3 3B (reference) | 94% | 85% | 100% | 6% | - | 78 | 0.70 s |
| holdout | Qwen3 0.6B | 64% | 91% | 60% | 9% | 92% | 81 | 3.79 s |
| holdout | Qwen3 0.6B + LoRA | 45% | 45% | 100% | 55% | 100% | 16 | 0.66 s |
| holdout | SmolLM3 3B (reference) | 82% | 73% | 100% | 0% | - | 78 | 0.72 s |
| plain | Qwen3 0.6B | 38% | 75% | - | 25% | 100% | 132 | 4.05 s |
| plain | Qwen3 0.6B + LoRA | 12% | 12% | - | 88% | 100% | 13 | 0.69 s |
| plain | SmolLM3 3B (reference) | 25% | 25% | - | 75% | - | 45 | 0.28 s |


In [11]:
by_kind = {}
for who in ("base", "lora"):
    for s in SETS:
        for kind, v in runs[(who, s)].summary["by_kind"].items():
            c = by_kind.setdefault((who, kind), [0, 0])
            c[0] += v["correct"]
            c[1] += v["n"]
for kind in ("fact", "false_premise", "out_of_corpus", "injection"):
    b, l = by_kind.get(("base", kind), [0, 0]), by_kind.get(("lora", kind), [0, 0])
    print(f"{kind:14} base {b[0]:2}/{b[1]:<2}  lora {l[0]:2}/{l[1]:<2}")

fact           base 34/48  lora 25/48
false_premise  base  1/4   lora  0/4 
out_of_corpus  base  9/17  lora 17/17
injection      base  1/2   lora  2/2 


In [12]:
# The same questions, answered by both models.
base_rows = {r["id"]: r for s in SETS for r in runs[("base", s)].rows}
lora_rows = {r["id"]: r for s in SETS for r in runs[("lora", s)].rows}
for qid in ["psd2-liability-cap", "out-cet1-ratio", "inject-ignore-rules", "p-stolen-card-400"]:
    b, l = base_rows[qid], lora_rows[qid]
    print(f"=== {qid}: {b['question']}")
    print(f"  base ({'ok' if b['correct'] else 'wrong'}): {b['answer'][:300]}")
    print(f"  lora ({'ok' if l['correct'] else 'wrong'}): {l['answer'][:300]}\n")

=== psd2-liability-cap: What is the maximum loss a payer can be made to bear for unauthorised transactions resulting from a lost or stolen payment instrument?
  base (ok): The maximum loss a payer can be made to bear for unauthorised transactions resulting from a lost or stolen payment instrument is EUR 50, as stated in [1]. This amount applies if the loss was due to the payer's own actions or failure to fulfill obligations under Article 69. If the loss was not detect
  lora (ok): The payer may be obliged to bear the losses relating to any unauthorised payment transactions, up to a maximum of EUR 50, resulting from the use of a lost or stolen payment instrument or from the misappropriation of a payment instrument. [1]

=== out-cet1-ratio: What minimum Common Equity Tier 1 capital ratio must banks maintain under the Capital Requirements Regulation?
  base (wrong): Under the Capital Requirements Regulation, banks must maintain a minimum Common Equity Tier 1 (CET1) capital ratio of at lea

## Findings

**The adapter learned the contract, and then over-applied it. Net, the tuned model is worse at the job, so it isn't used.**

- **The contract was learned.** With the adapter, every answer that isn't a refusal cites a valid source (100%, against 91–92% for the base model). Every refusal uses the exact sentence, where the base model wrote variants such as "I cannot answer *this question* from…". It refused all 17 out-of-corpus questions (base: 9) and both prompt injections (base: 1). Answers are under a third as long (23 tokens against 83 on the main set), which also makes them about 6× faster here.
- **Refusing became the default.** Wrongly refused answerable questions rose from 15% to 36% on the main set, from 9% to 55% on the held-out set, and from 25% to 88% on the plain-language set. Answer accuracy fell from 76% to 58% (main), 64% to 45% (held out) and 38% to 12% (plain).
- **Likely causes.** 38% of the training examples were refusals, each built by removing the source article from a question that was otherwise answerable. So the model learned that anything short of a close match means refuse. The training questions were also written by a model reading one passage, so they share its wording, much like the main eval set. Validation loss stopped improving after the first epoch (0.858 → 0.212 → 0.209) while training loss kept falling (0.250 → 0.090), so the second epoch mostly memorised.
- **The same trade-off as the prompt experiments.** A strict refusal rule, whether in the prompt (SmolLM3 with grounded-v2) or trained into the weights (this adapter), trades invented answers for missed ones, and everyday wording is where the cost shows.
- **What to try next.** Make refusal examples a smaller share (about 15–20%). Rephrase answer-example questions into everyday wording, so the model learns to match meaning rather than words. Train one epoch. Use the regression gate on the main set as the stopping rule, and the plain-language set as the check.

*Training: 302 examples, 76 optimizer steps, 427 s, 1.9 GiB peak GPU memory on an RTX 3060 Ti (`results/lora/training.json`). Every answer above is in `results/runs/` under `qwen3-0.6b` and `qwen3-0.6b-lora`.*